In [4]:
import os
import glob
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from collections import defaultdict, Counter
from torchvision import transforms, models
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from PIL import Image
import numpy as np
import warnings
warnings.filterwarnings('ignore')

import random

import pandas as pd

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [22]:
class CarColorDataset(Dataset):
    def __init__(self, file_paths, color_to_idx, transform=None):
        self.file_paths = file_paths
        self.color_to_idx = color_to_idx
        self.transform = transform

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        img_path = self.file_paths[idx]
        image = Image.open(img_path).convert('RGB')
        
        filename = os.path.basename(img_path)
        parts = filename.split('$$')
        
        if len(parts) >= 4:
            color = parts[3]
        else:
            color = 'Unknown'
            
        label = self.color_to_idx.get(color, 0)

        if self.transform:
            image = self.transform(image)

        return image, label

data_dir = 'confirmed_fronts'

all_files_raw = glob.glob(os.path.join(data_dir, '*', '*', '*.jpg'))

color_counts = Counter()
color_to_files = defaultdict(list)

for f in all_files_raw:
    parts = os.path.basename(f).split('$$')
    if len(parts) >= 4:
        color = parts[3]
        color_counts[color] += 1
        color_to_files[color].append(f)

TOP_N = 5
top_colors = [color for color, count in color_counts.most_common(TOP_N)]
print(f"{TOP_N} самых популярных цветов: {top_colors}")

sampled_files = []
random.seed(42)
MAX_PER_CLASS = 2000 

for color in top_colors:
    files = color_to_files[color]
    if len(files) > MAX_PER_CLASS:
        sampled_files.extend(random.sample(files, MAX_PER_CLASS))
    else:
        sampled_files.extend(files)

all_files = sampled_files 

unique_colors = set(top_colors)

color_to_idx = {color: idx for idx, color in enumerate(sorted(unique_colors))}
num_classes = len(color_to_idx)

train_files, test_files = train_test_split(all_files, test_size=0.2, random_state=42)

transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

transform_test = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_dataset = CarColorDataset(train_files, color_to_idx, transform=transform_train)
test_dataset = CarColorDataset(test_files, color_to_idx, transform=transform_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0, pin_memory=True)

print(f"Размер обучающей выборки (train): {len(train_dataset)} фото")
print(f"Размер тестовой выборки (test): {len(test_dataset)} фото")

5 самых популярных цветов: ['Black', 'Grey', 'White', 'Blue', 'Silver']
Размер обучающей выборки (train): 8000 фото
Размер тестовой выборки (test): 2000 фото


In [10]:
from tqdm.auto import tqdm

def train_model(model, train_loader, criterion, optimizer, epochs=5):
    model.train()
    for epoch in range(epochs):
        running_loss = 0.0
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]")
        
        for inputs, labels in pbar:
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * inputs.size(0)
            
            pbar.set_postfix({'loss': f"{loss.item():.4f}"})
            
        epoch_loss = running_loss / len(train_loader.dataset)
        print(f"Epoch {epoch+1} завершена - Average Loss: {epoch_loss:.4f}")

def evaluate_model(model, test_loader):
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        pbar = tqdm(test_loader, desc="Evaluation [Test]")
        for inputs, labels in pbar:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
    f1 = f1_score(all_labels, all_preds, average='macro')
    return f1

In [12]:
print("Model 1: ResNet-18 (предобучена на ImageNet)")
print(f"Текущее устройство: {device}") 

model_imagenet = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
model_imagenet.fc = nn.Linear(model_imagenet.fc.in_features, num_classes)
model_imagenet = model_imagenet.to(device)

criterion = nn.CrossEntropyLoss()
optimizer_imgnet = optim.Adam(model_imagenet.parameters(), lr=1e-4)

train_model(model_imagenet, train_loader, criterion, optimizer_imgnet, epochs=3)

print("оценка модели:")
f1_imagenet = evaluate_model(model_imagenet, test_loader)
print(f"F1 Macro (ImageNet ResNet-18): {f1_imagenet:.4f}\n")

Model 1: ResNet-18 (предобучена на ImageNet)
Текущее устройство: cuda


Epoch 1/3 [Train]: 100%|████████████████████████████████████████████████| 250/250 [00:59<00:00,  4.18it/s, loss=0.8472]


Epoch 1 завершена - Average Loss: 0.5787


Epoch 2/3 [Train]: 100%|████████████████████████████████████████████████| 250/250 [01:14<00:00,  3.38it/s, loss=0.1651]


Epoch 2 завершена - Average Loss: 0.3025


Epoch 3/3 [Train]: 100%|████████████████████████████████████████████████| 250/250 [01:06<00:00,  3.75it/s, loss=0.0510]


Epoch 3 завершена - Average Loss: 0.2051
оценка модели:


Evaluation [Test]: 100%|███████████████████████████████████████████████████████████████| 63/63 [00:29<00:00,  2.11it/s]

F1 Macro (ImageNet ResNet-18): 0.8361



In [13]:
import timm

print("Model 2: ResNet-50 (предобучена на ImageNet-21k)")

model_in21k = timm.create_model('resnetv2_50x1_bitm', pretrained=True, num_classes=num_classes)
model_in21k = model_in21k.to(device)

criterion = nn.CrossEntropyLoss()
optimizer_in21k = optim.Adam(model_in21k.parameters(), lr=1e-4)

train_model(model_in21k, train_loader, criterion, optimizer_in21k, epochs=3)

print("оценка модели:")
f1_in21k = evaluate_model(model_in21k, test_loader)
print(f"F1 Macro (ImageNet-21k ResNet-50): {f1_in21k:.4f}\n")

Model 2: ResNet-50 (предобучена на ImageNet-21k)


Epoch 1/3 [Train]: 100%|████████████████████████████████████████████████| 250/250 [02:13<00:00,  1.87it/s, loss=0.7904]


Epoch 1 завершена - Average Loss: 0.6447


Epoch 2/3 [Train]: 100%|████████████████████████████████████████████████| 250/250 [02:03<00:00,  2.03it/s, loss=0.1993]


Epoch 2 завершена - Average Loss: 0.3259


Epoch 3/3 [Train]: 100%|████████████████████████████████████████████████| 250/250 [01:52<00:00,  2.23it/s, loss=0.3684]


Epoch 3 завершена - Average Loss: 0.2452
оценка модели:


Evaluation [Test]: 100%|███████████████████████████████████████████████████████████████| 63/63 [00:12<00:00,  5.25it/s]

F1 Macro (ImageNet-21k ResNet-50): 0.8395



In [18]:
class ColorCNN(nn.Module):
    def __init__(self, num_classes):
        super(ColorCNN, self).__init__()
        
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(2, 2)
        
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(2, 2)
        
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.relu3 = nn.ReLU()
        self.pool3 = nn.MaxPool2d(2, 2)
        
        self.conv4 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm2d(256)
        self.relu4 = nn.ReLU()
        self.pool4 = nn.MaxPool2d(2, 2)

        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        
        self.fc = nn.Linear(256, num_classes)
        
    def forward(self, x):
        x = self.pool1(self.relu1(self.bn1(self.conv1(x))))
        x = self.pool2(self.relu2(self.bn2(self.conv2(x))))
        x = self.pool3(self.relu3(self.bn3(self.conv3(x))))
        x = self.pool4(self.relu4(self.bn4(self.conv4(x))))
        x = self.global_pool(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

In [19]:
model_custom = ColorCNN(num_classes).to(device)
optimizer_custom = optim.Adam(model_custom.parameters(), lr=1e-3)

train_model(model_custom, train_loader, criterion, optimizer_custom, epochs=5)
f1_custom = evaluate_model(model_custom, test_loader)
print(f"F1 Macro (Custom CNN): {f1_custom:.4f}\n")

Epoch 1/5 [Train]: 100%|████████████████████████████████████████████████| 250/250 [04:11<00:00,  1.00s/it, loss=1.0416]


Epoch 1 завершена - Average Loss: 1.0656


Epoch 2/5 [Train]: 100%|████████████████████████████████████████████████| 250/250 [03:12<00:00,  1.30it/s, loss=0.9229]


Epoch 2 завершена - Average Loss: 0.9172


Epoch 3/5 [Train]: 100%|████████████████████████████████████████████████| 250/250 [03:23<00:00,  1.23it/s, loss=0.7140]


...

Epoch 4 завершена - Average Loss: 0.7734


Epoch 5/5 [Train]: 100%|████████████████████████████████████████████████| 250/250 [03:12<00:00,  1.30it/s, loss=0.5415]


Epoch 5 завершена - Average Loss: 0.7164


Evaluation [Test]: 100%|███████████████████████████████████████████████████████████████| 63/63 [00:11<00:00,  5.42it/s]

F1 Macro (Custom CNN): 0.6313



Обоснование выбора:
цвет автомобиля - это глобальная характеристика изображения, не требующая глубокого понимания отдельных частей (колес, фар и т.д.). Для определения цвета можно извлечь низкоуровневые признаки (цветовые гистограммы, базовые текстуры). Слишком глубокая сеть при обучении с нуля на такой задаче будет переобучаться и тратить лишние вычислительные ресурсы. Поэтому лучше использовать легкую CNN: 4 сверточных блока с BatchNorm (для стабильности цвета при разном освещении) и Global Average Pooling в конце, чтобы агрегировать цветовые признаки со всего изображения перед полносвязным слоем.

In [20]:
results = {
    "Model": [
        "ResNet-18 (ImageNet-1k)", 
        "ResNet-50 (ImageNet-21k)", 
        "Custom CNN (без предобучения)"
    ],
    "F1_Macro": [
        f1_imagenet, 
        f1_in21k, 
        f1_custom
    ]
}

df_results = pd.DataFrame(results)

df_results.sort_values(by="F1_Macro", ascending=False, inplace=True)
df_results.reset_index(drop=True, inplace=True)

print("Итоговое сравнение моделей")
display(df_results)

Итоговое сравнение моделей


,Model,F1_Macro
0,ResNet-50 (ImageNet-21k),0.839479
1,ResNet-18 (ImageNet-1k),0.836128
2,Custom CNN (без предобучения),0.631263


обе предобученные модели показали результат примерно на 24% лучше, чем Custom CNN, которая обучалась со случайных весов.

почему так произошло: нейросети без предобучения (Custom CNN) на относительно небольшом датасете (10.000 фотографий) не хватает данных, чтобы научиться хорошо выделять контуры машин и понимать цвета. А ResNet-модели идут с готовым умением видеть мир, и им остаётся только сопоставить их знания с названиями цветов.

Также, если изначально не выделить небольшое кол-во цветов (у меня выделено 5), а оставить как есть, то будет около 23 цветов. из-за чего F1-Macro будет примерно в 2 раза ниже, потому что эта метрика считает качество для каждого класса отдельно, а потом усредняет их с одинаковым весом. в датасете встречаются редкие цвета (условно, один на несколько сотен или несколько тысяч). при усреднении F1-macro эти цвета уменьшают общий балл. Меньшее кол-во цветов снижает путаницу модели (то есть чем меньше цветов - тем меньше шанс того, что модель ошибется. так, в теории, она может спутать похожие по оттенку цвета)